# Paper 3 — E3b: Does the Temporal Dimension Localize?

The research question in its strong form: *can the UE exploit the temporal evolution of its CSI and
measurement reports — not just their instantaneous values — to localize itself within the serving
cluster?* Three nested feature sets on the M2b time-series datasets (four regimes, 19-20 UEs,
6000 samples at 5 ms):

- **F1 (snapshot):** the instantaneous per-array energy report (12 values, 1 dB quantized) — the
  E3 baseline.
- **F2 (+ window statistics):** F1 plus causal mean and standard deviation of each array's energy
  over the past 1 s — the temporal evolution of the *reports*.
- **F3 (+ adaptive-filtering innovation):** F2 plus the serving-link NLMS innovation (log EMA at
  the instant, plus its causal window mean/std) — the temporal evolution of the *CSI*, summarized
  by the filter the UE already runs.

Evaluation: kNN localization with a **leave-UEs-out split** (whole trajectories held out, never
random samples — a random split would leak adjacent samples between train and test and flatter
every feature set).

**Pre-registered expectations:** F2 < F1 (window statistics denoise the report and add a mobility
micro-signature). F3 vs F2 is the genuinely open question: if F3 improves materially (>10% median
in the dynamic regimes), the innovation is itself a *where* signal and the strong-form answer is
yes; if F3 ~ F2, the honest conclusion is that the innovation is a *when* signal (it decides when
to adapt, E1, and when to trust geometry, E3) but not a *where* signal — and the paper says so.

CPU-only, minutes.

In [ ]:
# %% [1] Setup
import os, json, uuid, glob
import numpy as np, h5py
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/paper3'
M2_DIR = os.path.join(BASE, 'm2')
E3B_DIR = os.path.join(BASE, 'e3b'); os.makedirs(E3B_DIR, exist_ok=True)

def drive_canary(path=E3B_DIR):
    token = str(uuid.uuid4()); c = os.path.join(path, '.canary')
    with open(c, 'w') as f: f.write(token)
    with open(c) as f: assert f.read() == token, 'DRIVE CANARY FAILED: phantom mount.'
    os.remove(c); print('Drive canary OK:', path)

drive_canary()

In [ ]:
# %% [2] Configuration
WIN = 200                       # causal window: 1 s at 5 ms
SUB = 20                        # evaluate every 100 ms
P_ORDER, MU_NLMS = 4, 0.5       # the deployable low-cost filter from E0/E1
EMA_ALPHA = 0.05
RSRP_Q = 1.0                    # dB
TEST_FRAC = 0.2                 # UEs held out (whole trajectories)
OUTAGE_MAX = 0.10               # drop UEs whose serving link is in outage >10% (as E1)
SEED = 2026
rng = np.random.default_rng(SEED)

In [ ]:
# %% [3] Load the four M2b datasets, drop outage UEs
DATA = {}
for p_ in sorted(glob.glob(os.path.join(M2_DIR, 'paper3_M2full_*.h5'))):
    with h5py.File(p_, 'r') as f:
        if 'energy_dB' not in f: continue
        E = f['energy_dB'][:]                        # [T, U, A]
        H = f['csi'][:]                              # [T, U, rx_ant, S] serving link
        pos = f['positions'][:]                      # [U, T, 3]
        serve = f['serving_array'][:].astype(np.int64)
        dt = float(f.attrs['fine_dt'])
    deficit = E.max(2) - np.take_along_axis(E, serve[None, :, None], 2)[:, :, 0]
    keep = (deficit > 20).mean(0) <= OUTAGE_MAX
    key = os.path.basename(p_).replace('paper3_M2full_', '').replace('.h5', '')
    key = key.replace('_U20_T600_seed2026', '')
    DATA[key] = dict(E=E[:, keep], H=H[:, keep], pos=pos[keep], dt=dt)
    print(f'{key:24s} | E {E[:, keep].shape} | dropped UEs: {int((~keep).sum())}')
assert DATA, 'no M2b datasets found'

In [ ]:
# %% [4] Feature construction (causal throughout)
def innovation_series(H_u):
    """Serving-link NLMS innovation ratio, EMA-smoothed; H_u [T, rx_ant, S]."""
    X = H_u.reshape(len(H_u), -1).astype(np.complex128)
    X = X/(np.sqrt((np.abs(X)**2).mean()) + 1e-30)
    K = X.shape[1]
    w = np.zeros((P_ORDER, K), np.complex128)
    ema = np.full(len(X), 1.0)
    e_prev = 1.0
    for t in range(P_ORDER, len(X)):
        u = X[t-P_ORDER:t][::-1]
        e = X[t] - (np.conj(w)*u).sum(0)
        inst = (np.abs(e)**2).sum()/((np.abs(X[t])**2).sum() + 1e-30)
        e_prev = (1-EMA_ALPHA)*e_prev + EMA_ALPHA*inst
        ema[t] = e_prev
        w += MU_NLMS*u*np.conj(e)/((np.abs(u)**2).sum(0) + 1e-6)
    ema[:P_ORDER] = ema[P_ORDER]
    return np.log10(ema + 1e-9)

def build_features(d):
    E, H, pos = d['E'], d['H'], d['pos']
    T, U, A = E.shape
    Eq = np.round(E/RSRP_Q)*RSRP_Q
    t_eval = np.arange(WIN, T, SUB)
    # causal window mean/std of the energies via cumulative sums
    c1 = np.cumsum(Eq, 0); c2 = np.cumsum(Eq**2, 0)
    wm = (c1[t_eval] - c1[t_eval-WIN])/WIN                       # [n, U, A]
    wv = np.maximum((c2[t_eval] - c2[t_eval-WIN])/WIN - wm**2, 0)
    ws = np.sqrt(wv)
    F1 = Eq[t_eval]                                              # [n, U, A]
    logI = np.stack([innovation_series(H[:, u]) for u in range(U)], 1)   # [T, U]
    i1 = np.cumsum(logI, 0); i2 = np.cumsum(logI**2, 0)
    im = (i1[t_eval] - i1[t_eval-WIN])/WIN
    isd = np.sqrt(np.maximum((i2[t_eval] - i2[t_eval-WIN])/WIN - im**2, 0))
    Ifeat = np.stack([logI[t_eval], im, isd], -1)                # [n, U, 3]
    y = np.transpose(pos[:, t_eval, :2], (1, 0, 2))              # [n, U, 2]
    sets = {
        'F1 snapshot':  F1,
        'F2 +window':   np.concatenate([F1, wm, ws], -1),
        'F3 +innovation': np.concatenate([F1, wm, ws, Ifeat], -1),
    }
    return sets, y

FEATS = {k: build_features(d) for k, d in DATA.items()}
print('features built:', {k: {n: v.shape for n, v in s.items()} for k, (s, _) in FEATS.items()})

In [ ]:
# %% [5] Leave-UEs-out kNN localization, per regime x feature set
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor

results = {}
for key, (sets, y) in FEATS.items():
    n, U, _ = y.shape
    te = rng.choice(U, max(2, int(round(TEST_FRAC*U))), replace=False)
    tr = np.setdiff1d(np.arange(U), te)
    row = {}
    for name, X in sets.items():
        Xtr = X[:, tr].reshape(-1, X.shape[-1]); ytr = y[:, tr].reshape(-1, 2)
        Xte = X[:, te].reshape(-1, X.shape[-1]); yte = y[:, te].reshape(-1, 2)
        mdl = make_pipeline(StandardScaler(), KNeighborsRegressor(5)).fit(Xtr, ytr)
        err = np.linalg.norm(mdl.predict(Xte) - yte, axis=1)
        row[name] = (float(np.median(err)), float(np.percentile(err, 90)))
    results[key] = row
    print(f'{key:24s} | test UEs {sorted(te.tolist())}')
    for name, (m, p90) in row.items():
        print(f'   {name:16s} median {m:6.2f} m | P90 {p90:6.2f} m')

In [ ]:
# %% [6] Pre-registered verdicts
print('Verdicts (per regime):')
for key, row in results.items():
    m1 = row['F1 snapshot'][0]; m2 = row['F2 +window'][0]; m3 = row['F3 +innovation'][0]
    v2 = 'PASS' if m2 < m1 else 'FAIL'
    gain3 = (m2 - m3)/m2 if m2 > 0 else 0.0
    v3 = 'WHERE-signal (>10%)' if gain3 > 0.10 else ('marginal' if gain3 > 0 else 'none')
    print(f'{key:24s} | [F2<F1] {v2} ({m1:.2f} -> {m2:.2f}) | '
          f'[F3 vs F2] {v3} ({m2:.2f} -> {m3:.2f}, {gain3*100:+.1f}%)')
print()
print('Interpretation key: if F3 adds nothing, the innovation is a WHEN signal (it gates E1 and')
print('the E3 hybrid) but not a WHERE signal — report it as such; both outcomes close E3b.')

drive_canary()
with open(os.path.join(E3B_DIR, 'e3b_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print('Saved e3b_results.json')